In [19]:
import kagglehub


In [20]:
path = kagglehub.dataset_download("chaitu20/ipl-dataset2008-2025")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ipl-dataset2008-2025' dataset.
Path to dataset files: /kaggle/input/ipl-dataset2008-2025


test 1

In [16]:
import os

# List the contents of the downloaded directory
print(os.listdir(path))
print(df.columns.tolist())
print(df.head(2))

['IPL.csv']
['Unnamed: 0', 'match_id', 'date', 'match_type', 'event_name', 'innings', 'batting_team', 'bowling_team', 'over', 'ball', 'ball_no', 'batter', 'bat_pos', 'runs_batter', 'balls_faced', 'bowler', 'valid_ball', 'runs_extras', 'runs_total', 'runs_bowler', 'runs_not_boundary', 'extra_type', 'non_striker', 'non_striker_pos', 'wicket_kind', 'player_out', 'fielders', 'runs_target', 'review_batter', 'team_reviewed', 'review_decision', 'umpire', 'umpires_call', 'player_of_match', 'match_won_by', 'win_outcome', 'toss_winner', 'toss_decision', 'venue', 'city', 'day', 'month', 'year', 'season', 'gender', 'team_type', 'superover_winner', 'result_type', 'method', 'balls_per_over', 'overs', 'event_match_no', 'stage', 'match_number', 'team_runs', 'team_balls', 'team_wicket', 'new_batter', 'power_surge_start', 'batter_runs', 'batter_balls', 'bowler_wicket', 'batting_partners', 'next_batter', 'striker_out', 'ball_in_over']
   Unnamed: 0  match_id        date match_type             event_name 

In [21]:
import pandas as pd
import os

# Load your Kaggle dataset, explicitly setting low_memory=False to handle mixed types
df = pd.read_csv(os.path.join(path, 'IPL.csv'), low_memory=False)

# Get every unique player name from both batting and bowling columns
all_players = pd.unique(df[['batter', 'bowler']].values.ravel('K'))
# Remove 'nan' if any
all_players = [name for name in all_players if str(name) != 'nan']

print(f"Total unique players to research: {len(all_players)}")

Total unique players to research: 788


In [30]:
from groq import Groq
client = Groq(api_key="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")

def get_ai_features(player_name, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            clean = re.sub(r'^```json\s*|```$', '',
                          response.choices[0].message.content.strip(),
                          flags=re.MULTILINE)
            return json.loads(clean)
        except Exception as e:
            print(f"  ↻ Retry {attempt+1} for {player_name}: {e}")
            time.sleep(3)
    return None

In [38]:
import json, time, re, os
import pandas as pd
from groq import Groq

# ── CONFIG ───────────────────────────────────────────────────────────────────
CHECKPOINT_FILE = 'ipl_master_players_temp.csv'
FINAL_FILE      = 'ipl_master_players_final.csv'
SAVE_EVERY      = 25
client          = Groq(api_key="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")

# ── DATA LOADING ─────────────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(path, 'IPL.csv'), low_memory=False)
all_players = pd.unique(df[['batter', 'bowler']].values.ravel('K'))
all_players = [name for name in all_players if str(name) != 'nan']

# ── HELPER ───────────────────────────────────────────────────────────────────
def parse_season(s):
    s = str(s).strip()
    return int(s.split('/')[0]) if '/' in s else int(s)

# ── KAGGLE-DERIVED FEATURES ──────────────────────────────────────────────────
def compute_kaggle_features(df):
    features = {}

    IPL_TEAMS = {
        "csk":  "Chennai Super Kings",
        "mi":   "Mumbai Indians",
        "rcb":  "Royal Challengers Bangalore",
        "kkr":  "Kolkata Knight Riders",
        "srh":  "Sunrisers Hyderabad",
        "dc":   "Delhi Capitals",
        "dd":   "Delhi Daredevils",
        "pbks": "Punjab Kings",
        "kxip": "Kings XI Punjab",
        "rr":   "Rajasthan Royals",
        "gl":   "Gujarat Lions",
        "rps":  "Rising Pune Supergiant",
        "ktr":  "Kochi Tuskers Kerala",
        "pwi":  "Pune Warriors India",
        "dh":   "Deccan Chargers",
        "gt":   "Gujarat Titans",
        "lsg":  "Lucknow Super Giants",
    }

    # Batting
    avg_position = df.groupby('batter')['bat_pos'].mean()
    balls_faced  = df.groupby('batter')['balls_faced'].sum()
    runs_scored  = df.groupby('batter')['runs_batter'].sum()
    strike_rate  = (runs_scored / balls_faced.replace(0, float('nan')) * 100).round(2).fillna(0)

    # Bowling
    death_balls        = df[df['over'] >= 17].groupby('bowler')['valid_ball'].sum()
    total_balls_bowled = df.groupby('bowler')['valid_ball'].sum()
    death_ratio        = (death_balls / total_balls_bowled).fillna(0)

    valid_dismissals = ['caught','bowled','lbw','stumped','caught and bowled','hit wicket']
    wickets      = df[df['wicket_kind'].isin(valid_dismissals)].groupby('bowler')['wicket_kind'].count()
    overs_bowled = df.groupby('bowler')['valid_ball'].sum() / 6
    runs_given   = df.groupby('bowler')['runs_bowler'].sum()
    economy      = (runs_given / overs_bowled.replace(0, float('nan'))).round(2).fillna(0)

    # Career
    seasons      = df.groupby('batter')['season'].apply(
                       lambda x: sorted([parse_season(s) for s in x.unique()]))
    teams_played = df.groupby('batter')['batting_team'].apply(lambda x: x.unique().tolist())

    # Franchise
    all_team_balls_counts = {
        key: df[df['batting_team'] == name].groupby('batter')['valid_ball'].sum()
        for key, name in IPL_TEAMS.items()
    }

    for player in pd.unique(df[['batter','bowler']].values.ravel('K')):
        if str(player) == 'nan':
            continue

        r   = int(runs_scored.get(player, 0))
        w   = int(wickets.get(player, 0))
        sr  = float(strike_rate.get(player, 0))
        eco = float(economy.get(player, 0))
        pos = float(avg_position.get(player, 99))
        dr  = float(death_ratio.get(player, 0))
        yrs = list(seasons.get(player, []))
        tms = list(teams_played.get(player, []))

        features[player] = {
            "total_runs":       r,
            "total_wickets":    w,
            "strike_rate":      sr,
            "economy_rate":     eco,
            "is_opener":        pos <= 2.0,
            "is_finisher":      pos >= 7.0 and r > 500,
            "is_death_bowler":  dr >= 0.4 and w > 10,
            "is_economical":    0 < eco < 7.5 and w > 10,
            "seasons_played":   str(yrs),
            "num_seasons":      len(yrs),
            "teams_played_for": str(tms),
            "num_teams":        len(tms),
            "one_franchise":    len(tms) == 1,
            "active_recent":    any(s >= 2022 for s in yrs) if yrs else False,
            "early_ipl_era":    any(s <= 2012 for s in yrs) if yrs else False,
            **{f"played_for_{key}": int(all_team_balls_counts[key].get(player, 0)) > 30
               for key in IPL_TEAMS}
        }
    return features

# ── AI FEATURES (Groq) ───────────────────────────────────────────────────────
def get_ai_features(player_name, retries=3):
    prompt = f"""Act as an IPL historian. For IPL player "{player_name}", return ONLY a valid JSON object.

{{
  "primary_role":        "Batsman | Bowler | All-rounder | Wicket-keeper",
  "secondary_role":      "e.g. Pinch-hitter, or null",
  "nationality":         "Indian | Overseas",
  "batting_hand":        "Right | Left",
  "bowling_style":       "Right-arm Fast | Left-arm Fast | Off-spin | Leg-spin | Left-arm spin | None",
  "is_wicketkeeper":     false,
  "is_captain":          false,
  "is_power_hitter":     false,
  "is_anchor":           false,
  "is_spinner":          false,
  "is_pacer":            false,
  "is_all_rounder":      false,
  "is_clutch":           false,
  "won_orange_cap":      false,
  "won_purple_cap":      false,
  "won_ipl_title":       false,
  "is_international_star": false,
  "personality_tag":     "One word: Aggressive | Calm | Finisher | Maverick | Consistent | Match-winner",
  "legacy_team":         "Team most identified with",
  "fan_perception":      "One sentence max",
  "iconic_moment":       "One sentence max"
}}

Rules:
- Pure JSON only. No markdown. No explanation.
- Booleans must be true or false (not strings).
- Use null if genuinely unknown."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            clean = re.sub(r'^```json\s*|```$', '',
                           response.choices[0].message.content.strip(),
                           flags=re.MULTILINE)
            return json.loads(clean)
        except Exception as e:
            err = str(e)
            if '429' in err:
                import re as re2
                match = re2.search(r'retry in (\d+\.?\d*)s', err)
                wait  = float(match.group(1)) + 5 if match else 60
                print(f"  ⏳ Rate limited. Waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"  ↻ Retry {attempt+1} for {player_name}: {e}")
                time.sleep(3)
    return None

# ── MAIN PIPELINE ────────────────────────────────────────────────────────────
def run_pipeline(df, all_players):
    print("Computing Kaggle features...")
    kaggle_features = compute_kaggle_features(df)

    try:
        done_df  = pd.read_csv(CHECKPOINT_FILE)
        done_set = set(done_df['player_name'].tolist())
        results  = done_df.to_dict('records')
        print(f"Resuming from checkpoint: {len(done_set)} already done.")
    except FileNotFoundError:
        done_set, results = set(), []

    remaining = [p for p in all_players if p not in done_set]
    print(f"Players to process: {len(remaining)}")

    for i, player in enumerate(remaining):
        print(f"[{i+1}/{len(remaining)}] {player}")

        ai_data = get_ai_features(player)
        if not ai_data:
            print(f"  ✗ Skipped {player}")
            continue

        k_data = kaggle_features.get(player, {
            "total_runs": 0, "total_wickets": 0,
            "strike_rate": 0.0, "economy_rate": 0.0,
            "is_opener": False, "is_finisher": False,
            "is_death_bowler": False, "is_economical": False,
            "seasons_played": "[]", "num_seasons": 0,
            "teams_played_for": "[]", "num_teams": 0,
            "one_franchise": False, "active_recent": False,
            "early_ipl_era": False,
        })

        k_data["seasons_played"]   = str(k_data.get("seasons_played", []))
        k_data["teams_played_for"] = str(k_data.get("teams_played_for", []))

        row = {"player_name": player}
        row.update(k_data)
        row.update(ai_data)
        results.append(row)

        if (i + 1) % SAVE_EVERY == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
            print(f"  ✓ Checkpoint saved ({len(results)} total)")

        time.sleep(2)  # Groq is fast, 0.5s is enough

    final_df = pd.DataFrame(results)
    final_df.fillna({"total_runs": 0, "total_wickets": 0,
                     "strike_rate": 0.0, "economy_rate": 0.0}, inplace=True)
    final_df.to_csv(FINAL_FILE, index=False)
    print(f"\nDone. {len(results)}/{len(all_players)} players → {FINAL_FILE}")
    print(f"Feature count per player: {final_df.shape[1]}")
    return final_df

# ── RUN ──────────────────────────────────────────────────────────────────────
master_df = run_pipeline(df, all_players)

KeyboardInterrupt: 

In [33]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('ipl_master_players_final.csv', '/content/drive/MyDrive/ipl_master_players_final.csv')
print("Saved to Drive!")

Mounted at /content/drive
Saved to Drive!


In [34]:
from google.colab import files
files.download('ipl_master_players_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json, time, re, os
import pandas as pd
from groq import Groq

# ── CONFIG ───────────────────────────────────────────────────────────────────
CHECKPOINT_FILE = 'ipl_master_players_temp.csv2'
FINAL_FILE      = 'ipl_master_players_final.csv2'
SAVE_EVERY      = 25
client          = Groq(api_key="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")

# ── DATA LOADING ─────────────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(path, 'IPL.csv'), low_memory=False)
all_players = pd.unique(df[['batter', 'bowler']].values.ravel('K'))
all_players = [name for name in all_players if str(name) != 'nan']

# ── HELPER ───────────────────────────────────────────────────────────────────
def parse_season(s):
    s = str(s).strip()
    return int(s.split('/')[0]) if '/' in s else int(s)

# ── KAGGLE-DERIVED FEATURES ──────────────────────────────────────────────────
def compute_kaggle_features(df):
    features = {}

    IPL_TEAMS = {
        "csk":  "Chennai Super Kings",
        "mi":   "Mumbai Indians",
        "rcb":  "Royal Challengers Bangalore",
        "kkr":  "Kolkata Knight Riders",
        "srh":  "Sunrisers Hyderabad",
        "dc":   "Delhi Capitals",
        "dd":   "Delhi Daredevils",
        "pbks": "Punjab Kings",
        "kxip": "Kings XI Punjab",
        "rr":   "Rajasthan Royals",
        "gl":   "Gujarat Lions",
        "rps":  "Rising Pune Supergiant",
        "ktr":  "Kochi Tuskers Kerala",
        "pwi":  "Pune Warriors India",
        "dh":   "Deccan Chargers",
        "gt":   "Gujarat Titans",
        "lsg":  "Lucknow Super Giants",
    }

    # Batting
    avg_position = df.groupby('batter')['bat_pos'].mean()
    balls_faced  = df.groupby('batter')['balls_faced'].sum()
    runs_scored  = df.groupby('batter')['runs_batter'].sum()
    strike_rate  = (runs_scored / balls_faced.replace(0, float('nan')) * 100).round(2).fillna(0)

    # Bowling
    death_balls        = df[df['over'] >= 17].groupby('bowler')['valid_ball'].sum()
    total_balls_bowled = df.groupby('bowler')['valid_ball'].sum()
    death_ratio        = (death_balls / total_balls_bowled).fillna(0)

    valid_dismissals = ['caught','bowled','lbw','stumped','caught and bowled','hit wicket']
    wickets      = df[df['wicket_kind'].isin(valid_dismissals)].groupby('bowler')['wicket_kind'].count()
    overs_bowled = df.groupby('bowler')['valid_ball'].sum() / 6
    runs_given   = df.groupby('bowler')['runs_bowler'].sum()
    economy      = (runs_given / overs_bowled.replace(0, float('nan'))).round(2).fillna(0)

    # Career
    seasons      = df.groupby('batter')['season'].apply(
                       lambda x: sorted([parse_season(s) for s in x.unique()]))
    teams_played = df.groupby('batter')['batting_team'].apply(lambda x: x.unique().tolist())

    # Franchise
    all_team_balls_counts = {
        key: df[df['batting_team'] == name].groupby('batter')['valid_ball'].sum()
        for key, name in IPL_TEAMS.items()
    }

    for player in pd.unique(df[['batter','bowler']].values.ravel('K')):
        if str(player) == 'nan':
            continue

        r   = int(runs_scored.get(player, 0))
        w   = int(wickets.get(player, 0))
        sr  = float(strike_rate.get(player, 0))
        eco = float(economy.get(player, 0))
        pos = float(avg_position.get(player, 99))
        dr  = float(death_ratio.get(player, 0))
        yrs = list(seasons.get(player, []))
        tms = list(teams_played.get(player, []))

        features[player] = {
            "total_runs":       r,
            "total_wickets":    w,
            "strike_rate":      sr,
            "economy_rate":     eco,
            "is_opener":        pos <= 2.0,
            "is_finisher":      pos >= 7.0 and r > 500,
            "is_death_bowler":  dr >= 0.4 and w > 10,
            "is_economical":    0 < eco < 7.5 and w > 10,
            "seasons_played":   str(yrs),
            "num_seasons":      len(yrs),
            "teams_played_for": str(tms),
            "num_teams":        len(tms),
            "one_franchise":    len(tms) == 1,
            "active_recent":    any(s >= 2022 for s in yrs) if yrs else False,
            "early_ipl_era":    any(s <= 2012 for s in yrs) if yrs else False,
            **{f"played_for_{key}": int(all_team_balls_counts[key].get(player, 0)) > 30
               for key in IPL_TEAMS}
        }
    return features

# ── AI FEATURES (Groq) ───────────────────────────────────────────────────────
def get_ai_features(player_name, retries=5):
    prompt = f"""Act as a strict IPL cricket historian with deep knowledge.
For IPL player "{player_name}", return ONLY a valid JSON object.

STRICTNESS RULES FOR EVERY FIELD:
- Only set true if you are CERTAIN based on widely known IPL record.
- If you have any doubt, use false or null.
- Do NOT guess. Do NOT infer. Only use well-established facts.

For confidence fields: 1.0 = absolutely certain, 0.5 = reasonable belief, 0.3 = uncertain.

{{
  "primary_role":        "Batsman | Bowler | All-rounder | Wicket-keeper",
  "secondary_role":      "Pinch-hitter | Death-hitter | Floater | null — only if widely known",
  "nationality":         "Indian | Overseas",
  "batting_hand":        "Right | Left",
  "bowling_style":       "Right-arm Fast | Right-arm Medium | Left-arm Fast | Left-arm Medium | Off-spin | Leg-spin | Left-arm spin | None",

  "is_wicketkeeper":     false,
  "is_captain":          false,
  "is_international_star": false,

  "is_all_rounder":      {{ "value": false, "confidence": 0.0 }},
  "is_power_hitter":     {{ "value": false, "confidence": 0.0 }},
  "is_anchor":           {{ "value": false, "confidence": 0.0 }},
  "is_finisher":         {{ "value": false, "confidence": 0.0 }},
  "is_spinner":          {{ "value": false, "confidence": 0.0 }},
  "is_pacer":            {{ "value": false, "confidence": 0.0 }},
  "is_death_bowler":     {{ "value": false, "confidence": 0.0 }},
  "is_clutch":           {{ "value": false, "confidence": 0.0 }},

  "won_orange_cap":      false,
  "won_purple_cap":      false,
  "won_ipl_title":       false,

  "personality_tag":     "ONE word only: Aggressive | Calm | Finisher | Maverick | Consistent | Explosive | Crafty | null",
  "legacy_team":         "Full team name they are MOST identified with in IPL history, or null",
  "fan_perception":      "One factual sentence about how IPL fans remember this player. null if obscure player.",
  "iconic_moment":       "One sentence: their single most famous IPL moment. null if none known."
}}

STRICT RULES:
- Pure JSON only. No markdown. No explanation. No extra keys.
- Booleans: true or false only (never strings).
- Confidence: float between 0.0 and 1.0.
- Set is_all_rounder true ONLY if widely recognized as a genuine IPL all-rounder who both bats and bowls significantly.
- Set is_clutch true ONLY if player has a strong public reputation for performing under pressure.
- Set is_finisher true ONLY if player is specifically known for finishing IPL innings.
- Set won_orange_cap / won_purple_cap true ONLY if they actually won it in any IPL season.
- legacy_team, fan_perception, iconic_moment MUST be filled if player is well-known. null only for obscure players."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0       # zero temp = most deterministic
            )
            raw  = response.choices[0].message.content.strip()
            # Strip markdown fences
            clean = re.sub(r'^```json\s*|^```\s*|```$', '', raw, flags=re.MULTILINE).strip()
            data  = json.loads(clean)

            # ── Flatten confidence fields into separate columns ──────────────
            # Converts {"value": true, "confidence": 0.9}
            # into  is_clutch=True, is_clutch_conf=0.9
            flat = {}
            for key, val in data.items():
                if isinstance(val, dict) and 'value' in val and 'confidence' in val:
                    flat[key]              = val['value']
                    flat[f"{key}_conf"]    = float(val['confidence'])
                else:
                    flat[key] = val

            # ── Validate required fields are not missing ─────────────────────
            required = ['primary_role', 'nationality', 'legacy_team','fan_perception', 'iconic_moment']
            missing  = [f for f in required if flat.get(f) is None]
            required_core = ['primary_role', 'nationality']
            required_semantic = ['legacy_team', 'fan_perception', 'iconic_moment']
            missing_core = [f for f in required_core if flat.get(f) is None]
            missing_semantic = [f for f in required_semantic if flat.get(f) is None]

            if missing_core and attempt < retries - 1:
              print(f"  ⚠ Missing core {missing_core} for {player_name}, retrying...")
              time.sleep(2)
              continue
            if missing and attempt < retries - 1:
                print(f"  ⚠ Missing {missing} for {player_name}, retrying...")
                time.sleep(2)
                continue
            if missing_semantic:
                print(f"  ℹ {player_name} is obscure, accepting nulls.")

            return flat

        except json.JSONDecodeError as e:
            print(f"  ↻ JSON parse fail attempt {attempt+1} for {player_name}: {e}")
            time.sleep(2)
        except Exception as e:
            err = str(e)
            if '429' in err:
                match = re.search(r'retry in (\d+\.?\d*)s', err)
                wait  = float(match.group(1)) + 5 if match else 60
                print(f"  ⏳ Rate limited. Waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"  ↻ Error attempt {attempt+1} for {player_name}: {e}")
                time.sleep(3)
    return None

# ── MAIN PIPELINE ────────────────────────────────────────────────────────────
def run_pipeline(df, all_players):
    print("Computing Kaggle features...")
    kaggle_features = compute_kaggle_features(df)

    try:
        done_df  = pd.read_csv(CHECKPOINT_FILE)
        done_set = set(done_df['player_name'].tolist())
        results  = done_df.to_dict('records')
        print(f"Resuming from checkpoint: {len(done_set)} already done.")
    except FileNotFoundError:
        done_set, results = set(), []

    remaining = [p for p in all_players if p not in done_set]
    print(f"Players to process: {len(remaining)}")

    for i, player in enumerate(remaining):
        print(f"[{i+1}/{len(remaining)}] {player}")

        ai_data = get_ai_features(player)
        if not ai_data:
            print(f"  ✗ Skipped {player}")
            continue

        k_data = kaggle_features.get(player, {
            "total_runs": 0, "total_wickets": 0,
            "strike_rate": 0.0, "economy_rate": 0.0,
            "is_opener": False, "is_finisher": False,
            "is_death_bowler": False, "is_economical": False,
            "seasons_played": "[]", "num_seasons": 0,
            "teams_played_for": "[]", "num_teams": 0,
            "one_franchise": False, "active_recent": False,
            "early_ipl_era": False,
        })

        k_data["seasons_played"]   = str(k_data.get("seasons_played", []))
        k_data["teams_played_for"] = str(k_data.get("teams_played_for", []))

        row = {"player_name": player}
        row.update(k_data)
        row.update(ai_data)
        results.append(row)

        if (i + 1) % SAVE_EVERY == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
            print(f"  ✓ Checkpoint saved ({len(results)} total)")

        time.sleep(2)  # Groq is fast, 0.5s is enough

    final_df = pd.DataFrame(results)
    final_df.fillna({"total_runs": 0, "total_wickets": 0,
                     "strike_rate": 0.0, "economy_rate": 0.0}, inplace=True)
    final_df.to_csv(FINAL_FILE, index=False)
    print(f"\nDone. {len(results)}/{len(all_players)} players → {FINAL_FILE}")
    print(f"Feature count per player: {final_df.shape[1]}")
    return final_df

# ── RUN ──────────────────────────────────────────────────────────────────────
master_df = run_pipeline(df, all_players)

Computing Kaggle features...
Players to process: 788
[1/788] SC Ganguly
[2/788] BB McCullum
[3/788] RT Ponting
[4/788] DJ Hussey
[5/788] Mohammad Hafeez
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[6/788] R Dravid
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[7/788] W Jaffer
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[8/788] V Kohli
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[9/788] JH Kallis
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[10/788] CL White
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[11/788] MV Boucher
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
[12/788] B Akhil
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s...
  ⏳ Rate limited. Waiting 60s..